# Experiment Dashboard

실험별 모델 설정, feature 설정, CV/holdout 결과를 비교하는 노트북입니다.

- registry source: `analysis/experiment_registry.csv`
- 실험이 추가될수록 자동 누적됩니다.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
REGISTRY_PATH = Path('analysis/experiment_registry.csv')

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(REGISTRY_PATH)

runs = pd.read_csv(REGISTRY_PATH)
runs = runs.sort_values('timestamp_utc', ascending=False).reset_index(drop=True)
print('n_runs:', len(runs))
runs.head()

In [ ]:
cols = [
    'timestamp_utc',
    'run_id',
    'model_name',
    'split_batch',
    'n_features',
    'reg_lambda',
    'cv_rmse_mean',
    'cv_mae_mean',
    'cv_r2_mean',
    'holdout_rmse',
    'holdout_mae',
    'holdout_r2',
]
runs[cols]

In [ ]:
runs.groupby(['model_name', 'split_batch'])[['cv_rmse_mean', 'holdout_rmse', 'holdout_r2']].agg(['count', 'min', 'mean', 'max'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(runs['n_features'], runs['holdout_rmse'])
axes[0].set_title('Holdout RMSE vs #Features')
axes[0].set_xlabel('n_features')
axes[0].set_ylabel('holdout_rmse')

axes[1].scatter(runs['reg_lambda'], runs['holdout_rmse'])
axes[1].set_title('Holdout RMSE vs Regularization')
axes[1].set_xlabel('reg_lambda')
axes[1].set_ylabel('holdout_rmse')
axes[1].set_xscale('log')

axes[2].scatter(runs['cv_rmse_mean'], runs['holdout_rmse'])
axes[2].set_title('CV RMSE vs Holdout RMSE')
axes[2].set_xlabel('cv_rmse_mean')
axes[2].set_ylabel('holdout_rmse')

plt.tight_layout()
plt.show()

In [ ]:
latest = runs.iloc[0].copy()
feature_config = json.loads(latest['feature_config']) if isinstance(latest['feature_config'], str) and latest['feature_config'] else {}
cv_folds = json.loads(latest['cv_folds']) if isinstance(latest['cv_folds'], str) and latest['cv_folds'] else []

print('Latest run_id:', latest['run_id'])
print('Latest model:', latest['model_name'])
print('Latest split batch:', latest['split_batch'])
print('Latest holdout RMSE:', latest['holdout_rmse'])
print('Latest holdout R2:', latest['holdout_r2'])
print('\nFeature config:')
feature_config

In [ ]:
pd.DataFrame(cv_folds)